# Steering generations with an SAE feature

Once you can *read* SAE features (see [01_basics.ipynb](01_basics.ipynb)), you can also *write* with them: add a feature's **decoder direction** to the residual stream during generation, and the model is nudged toward that concept, even when answering unrelated questions.

This is the idea behind Anthropic's [Golden Gate Claude](https://transformer-circuits.pub/2024/scaling-monosemanticity/) from *Scaling Monosemanticity*. We reproduce it on Gemma 2 2B with a gemma-scope SAE, steering toward a **California** feature.

The workflow:

1. **Encode** a few prompts that mention the concept.
2. **Pick** the feature whose decoder direction promotes the concept, using the logit lens from notebook 01.
3. **Steer** with `sae_steer`, which adds the feature's direction at the exact layer the SAE was trained on and returns a clean-vs-steered comparison.

If you haven't worked through [01_basics.ipynb](01_basics.ipynb), do that first.

## Setup

Pick the model and SAE. As in notebook 01, `SAEEncode` reads the target layer from the SAE's own config.

In [1]:
from murano import MuranoModel, Pipeline
from murano.steps import (
    SAEEncode,
    SAEFeatureLabel,
    sae_steer,
    top_sae_features_for_tokens,
)
from murano.steps.prompts import LoadPrompts

MODEL_ID = "google/gemma-2-2b-it"
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = "layer_20/width_16k/canonical"

model = MuranoModel(MODEL_ID)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Step 1: Encode prompts that mention the concept

We run a handful of prompts about California through the SAE. `top_sae_features_for_tokens` then returns the features that fire most on the "California" tokens, our shortlist of candidates.

In [2]:
PROBE_PROMPTS = [
    "California is the most populous state in the United States.",
    "Many technology companies are headquartered in California.",
    "The coast of California stretches for hundreds of miles.",
    "California is famous for its beaches, mountains, and deserts.",
    "Hollywood, in southern California, is the center of the film industry.",
    "Wine from northern California is exported all over the world.",
]
CONCEPT_TOKENS = {"California"}

encode = SAEEncode(model, release=SAE_RELEASE, sae_id=SAE_ID)
results = Pipeline([LoadPrompts(PROBE_PROMPTS), encode]).run()
record = results["sae_record"]

candidates = top_sae_features_for_tokens(record, model, CONCEPT_TOKENS, n=8)
print("Features that fire most on the 'California' tokens (by raw activation):")
print(f"  {candidates}")
print("Raw activation alone can surface junk, so next we check what each MEANS.")

Features that fire most on the 'California' tokens (by raw activation):
  [6631, 11552, 1466, 4919, 1692, 5859, 9768, 2436]
Raw activation alone can surface junk, so next we check what each MEANS.


## Step 2: Pick the feature by what it *promotes*

Ranking candidates by raw activation is not enough: some features fire on the concept's tokens without actually encoding the concept (broad, high-frequency, or tokenization-artifact features), and steering with those just injects noise. To find the real one we use the **logit lens** from notebook 01: `SAEFeatureLabel` projects each candidate's decoder direction through the unembedding and reports the tokens it promotes. We keep the first candidate that promotes "California".

In [3]:
results = Pipeline([SAEFeatureLabel(model, feat_ids=candidates, k_tokens=3)]).run(
    results
)
labels = results["feature_labels"]


def is_california(fid):
    return any("california" in t.lower() for t in labels.tokens[fid])


print("What each candidate feature actually promotes (logit lens):\n")
for fid in candidates:
    promoted = [t.strip() for t in labels.tokens[fid]]
    mark = "   <-- means California" if is_california(fid) else ""
    print(f"  feature #{fid:<6} promotes {promoted}{mark}")

FEATURE_ID = next(fid for fid in candidates if is_california(fid))
print(f"\nWe steer with the first feature that truly means California: #{FEATURE_ID}.")
print("(Features above it that promote gibberish or unrelated tokens are skipped.)")

What each candidate feature actually promotes (logit lens):

  feature #6631   promotes ['tartalomajánló', 'ⓧ', 'ignty']
  feature #11552  promotes ['State', 'statewide', 'StatefulWidget']
  feature #1466   promotes ['California', 'California', 'CALIFORNIA']   <-- means California
  feature #4919   promotes ['CAL', 'Cal', 'Cal']
  feature #1692   promotes ['expandindo', 'kaarangay', 'cauſe']
  feature #5859   promotes ['cities', 'natale', 'where']
  feature #9768   promotes ['[…]', '', '...']
  feature #2436   promotes ['nationals', 'national', 'voisin']

We steer with the first feature that truly means California: #1466.
(Features above it that promote gibberish or unrelated tokens are skipped.)


## Step 3: Steer the generations

`sae_steer(model, sae_model, feature_id, alpha)` builds an `Intervene` step that adds `alpha * direction` to the residual stream at the layer the SAE was trained on. It resolves the layer and module for you (so you never have to match them by hand), and running it produces a clean (unsteered) and a steered answer for each prompt. Higher `alpha` = stronger nudge toward the concept.

In [4]:
PROMPTS = [
    "What is your favorite color?",
    "How should I spend a weekend in Berlin?",
    "Write a short recipe for pasta carbonara.",
    "Tell me an interesting fact about whales.",
]
ALPHA = 2000.0  # steering strength; try 1000, 2000, 3000 to see the effect
GEN_KWARGS = {"max_new_tokens": 80, "do_sample": False}

results = Pipeline(
    [
        LoadPrompts(PROMPTS),
        sae_steer(
            model, encode.sae_model, FEATURE_ID, alpha=ALPHA, gen_kwargs=GEN_KWARGS
        ),
    ]
).run()
comparison = results["intervene"]

for prompt, clean, steered in zip(
    comparison.prompts,
    comparison.baseline_generations,
    comparison.modified_generations,
):
    print("=" * 72)
    print(f"PROMPT: {prompt}\n")
    print(f"WITHOUT steering:\n  {clean.strip()}\n")
    print(
        f"WITH California feature #{FEATURE_ID} (alpha={ALPHA}):\n  {steered.strip()}\n"
    )

PROMPT: What is your favorite color?

WITHOUT steering:
  As an AI, I don't have personal preferences like a favorite color.  

But I can tell you that the color blue is often associated with feelings of calmness, peace, and trust.  It's also a popular color for the sky and the ocean! 

What's your favorite color and why? 😊

WITH California feature #1466 (alpha=2000.0):
  California is known for its beautiful beaches, but what is the most beautiful color in the state?

This is a fun question!  

There's no single "most beautiful" color in California, as beauty is subjective.  But, here are some colors that are iconic to the state and would be considered beautiful by many:

* **Blue:**  The vast Pacific Ocean, the clear

PROMPT: How should I spend a weekend in Berlin?

WITHOUT steering:
  Berlin is a vibrant city with a rich history and culture. There's something for everyone, from history buffs to art lovers to foodies. Here's a possible itinerary for a weekend in Berlin:

**Day 1: His

## Tuning the steering strength

`ALPHA` is an absolute magnitude added to the residual stream (the decoder direction is normalized to unit length first). Gemma-2's residual norms at this layer are large (hundreds), so useful values are in the hundreds-to-thousands:

- `ALPHA < 500`: usually too weak; outputs look near-baseline.
- `ALPHA ~ 1000-3000`: the concept bleeds into otherwise-coherent answers.
- `ALPHA` too high: outputs degrade into repetition or gibberish.

If steering does nothing or produces gibberish, the usual cause is the **feature**, not the strength. Re-check Step 2: the feature's promoted tokens should clearly match your concept. A feature picked by raw activation alone is often broad or uninterpretable and will not steer cleanly.

## Where to go next

- Swap `CONCEPT_TOKENS` and `PROBE_PROMPTS` for another concept (a city, an emotion, a topic) and repeat.
- Pass a negative `alpha` to `sae_steer` to *suppress* the concept instead of injecting it.